### 매뉴얼 - 다중 선형 회귀 (Multiple Linear Regression)

본 노트북은 AI 컴패니언 의존도(정서적 애착도 `emotional_attachment_score`)를 다중 수치형 피처 및 원-핫 인코딩된 범주형 피처들을 포함하여 다중 선형 회귀 모델로 예측하고 분석한다.

In [ ]:
import os
import platform
import pandas as pd
import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정
if platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
elif platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
else:
    plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

# 1. 데이터 로드
data = pd.read_csv('data/ai_companion_dependency_dataset.csv')

# 2. 데이터 필터링 (행: 15~24세 및 주요 이용목적 3종)
target_col = ['Companionship', 'Entertainment', 'Therapy-Substitute']
df_filtered = data[
    (data['age'] >= 15) & (data['age'] <= 24) &
    (data['primary_use_case'].isin(target_col))
].copy()

# 3. 피처 선택 (열)
selected_features = [
    'age', 'education_level', 'income_level', 'primary_use_case',
    'daily_ai_chat_hours', 'human_social_interaction_hours', 'social_media_hours_daily',
    'screen_time_total_hours', 'number_of_close_friends', 'real_relationship_satisfaction',
    'sleep_hours', 'sleep_quality_score', 'exercise_hours_weekly',
    'loneliness_score', 'anxiety_score', 'depression_score', 'stress_score',
    'self_esteem_score', 'therapy_attendance',
    'emotional_attachment_score'
]
df = df_filtered[selected_features].reset_index(drop=True)
print(f"분석 데이터 크기: {df.shape}")


In [ ]:
# ===========================================================================================================
# 전처리: 범주형 피처 원-핫 인코딩 (One-Hot Encoding)
# ===========================================================================================================
categorical_cols = ['education_level', 'income_level', 'primary_use_case', 'therapy_attendance']

# 범주형 변수 One-Hot Encoding 적용 (다중공선성 방지를 위해 drop_first=True)
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)

print(f"원-핫 인코딩 후 데이터 크기: {df_encoded.shape}")
display(df_encoded.head())


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ===========================================================================================================
# 5. 데이터 분할 & 다공선성(VIF) 확인
# ===========================================================================================================
X = df_encoded.drop(columns=['emotional_attachment_score'])
y = df_encoded['emotional_attachment_score']

# 자체 VIF 계산 함수 구현 (LinearRegression R^2 이용)
def calculate_vif(df_features):
    vif_list = []
    cols = df_features.columns
    for col in cols:
        y_sub = df_features[col]
        X_sub = df_features.drop(columns=[col])
        r2_sub = LinearRegression().fit(X_sub, y_sub).score(X_sub, y_sub)
        vif = 1.0 / (1.0 - r2_sub) if r2_sub < 1.0 else np.inf
        vif_list.append(vif)
    return pd.DataFrame({'Feature': cols, 'VIF': vif_list})

vif_df = calculate_vif(X)
print("--- [독립변수 VIF (분산팽창지수)] ---")
print(vif_df.sort_values(by='VIF', ascending=False).to_string(index=False))

# Train/Test 분할 (80:20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\n훈련 데이터: {X_train.shape}, 테스트 데이터: {X_test.shape}")


In [ ]:
# ===========================================================================================================
# 6. 모델 선택, 7. 학습 및 8. 평가
# ===========================================================================================================
mlr_model = LinearRegression()
mlr_model.fit(X_train, y_train)

# 예측
y_train_pred = mlr_model.predict(X_train)
y_test_pred = mlr_model.predict(X_test)

# 성능 평가
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_mae = mean_absolute_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

# 수정된 R² (Adjusted R²)
n_test = len(y_test)
p_test = X_test.shape[1]
adj_test_r2 = 1 - (1 - test_r2) * (n_test - 1) / (n_test - p_test - 1)

print("\n--- [다중 선형 회귀 모델 평가 결과] ---")
print(f"[Train 데이터] MSE: {train_mse:.4f} | RMSE: {train_rmse:.4f} | MAE: {train_mae:.4f} | R²: {train_r2:.4f}")
print(f"[Test  데이터] MSE: {test_mse:.4f} | RMSE: {test_rmse:.4f} | MAE: {test_mae:.4f} | R²: {test_r2:.4f} | Adj R²: {adj_test_r2:.4f}")

# 회귀 계수(Coefficients) 시각화
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': mlr_model.coef_
}).sort_values(by='Coefficient', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(data=coef_df, x='Coefficient', y='Feature', palette='vlag')
plt.title('다중 선형 회귀 모델 피처 가중치 (Coefficients)', fontsize=14, pad=12)
plt.axvline(0, color='black', linestyle='--', linewidth=1)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 실제값 vs 예측값 산점도 시각화
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_test_pred, color='#3498db', alpha=0.7, edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Ideal Fit (y=x)')
plt.title('다중 선형 회귀: 실제 정서적 애착도 vs 예측값', fontsize=14)
plt.xlabel('실제값 (Actual)', fontsize=12)
plt.ylabel('예측값 (Predicted)', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
